# Pump it Up: Data Mining the Water Table

### Objective

The goal is to predict the **operating condition of a waterpoint** for each record in the dataset.

### Data Dictionary

The available variables are the following:
<table>
    <thead>
        <tr>
            <th>Variable</th><th>Description</th>
            <th>Variable</th><th>Description</th>
            <th>Variable</th><th>Description</th>
        </tr>
    </thead>
    <tbody>
        <tr><td>amount_tsh</td><td>Total static head</td><td>date_recorded</td><td>Date recorded</td><td>funder</td><td>Who funded the well</td></tr>
        <tr><td>gps_height</td><td>Altitude of the well</td><td>installer</td><td>Installer</td><td>longitude</td><td>GPS longitude</td></tr>
        <tr><td>latitude</td><td>GPS latitude</td><td>wpt_name</td><td>Waterpoint name</td><td>num_private</td><td>Private indicator</td></tr>
        <tr><td>basin</td><td>Geographic water basin</td><td>subvillage</td><td>Sub-village</td><td>region</td><td>Geographic region</td></tr>
        <tr><td>region_code</td><td>Region code</td><td>district_code</td><td>District code</td><td>lga</td><td>Local government area</td></tr>
        <tr><td>ward</td><td>Administrative ward</td><td>population</td><td>Population around the well</td><td>public_meeting</td><td>Public meeting</td></tr>
        <tr><td>recorded_by</td><td>Recorded by</td><td>scheme_management</td><td>Operator</td><td>scheme_name</td><td>Scheme name</td></tr>
        <tr><td>permit</td><td>Has a permit</td><td>construction_year</td><td>Construction year</td><td>extraction_type</td><td>Extraction type</td></tr>
        <tr><td>extraction_type_group</td><td>Extraction group</td><td>extraction_type_class</td><td>Extraction class</td><td>management</td><td>Management</td></tr>
        <tr><td>management_group</td><td>Management group</td><td>payment</td><td>Water cost</td><td>payment_type</td><td>Payment type</td></tr>
        <tr><td>water_quality</td><td>Water quality</td><td>quality_group</td><td>Quality group</td><td>quantity</td><td>Water quantity</td></tr>
        <tr><td>quantity_group</td><td>Quantity group</td><td>source</td><td>Water source</td><td>source_type</td><td>Source type</td></tr>
        <tr><td>source_class</td><td>Source class</td><td>waterpoint_type</td><td>Waterpoint type</td><td>waterpoint_type_group</td><td>Waterpoint type group</td></tr>
    </tbody>
</table>

**Target Variable**

The target labels are simple and have **three possible values**:

- **functional**: The waterpoint is operational and needs no repairs.
- **functional needs repair**: The waterpoint is operational but needs repairs.
- **non functional**: The waterpoint is not operational.

**Dataset from the DrivenData competition**:
[Pump It Up: Data Mining the Water Table](https://www.drivendata.org/competitions/7/pump-it-up-data-mining-the-water-table/)

*Comments in blue record my thought process during the project.*
*Regular markdown cells document how the model was built.*

### Libraries

In [28]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from category_encoders import TargetEncoder

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

### Data Loading

Data is loaded, checked for duplicates, and the DataFrames are merged.

In [29]:
# Load data
train = pd.read_csv("train.csv")
labels = pd.read_csv("labels.csv")
test  = pd.read_csv("test.csv")

In [30]:
# Check for duplicates
print(train.id.nunique())
print(labels.id.nunique())

59400
59400


No duplicate ids were found.

In [31]:
# Merge data on 'id'
train_merge = pd.merge(train, labels, on='id')

# Add a column to tell them apart
train_merge['dataset'] = 'train'
test['dataset'] = 'test'

# Concatenate into a single DataFrame
df = pd.concat([train_merge, test], axis=0, ignore_index=True)

# DataFrame dimensions
print(f"DataFrame shape: {df.shape}")
df.head()

DataFrame shape: (74250, 42)


,id,amount_tsh,date_recorded,funder,gps_height,installer,longitude,latitude,wpt_name,num_private,...,quality_group,quantity,quantity_group,source,source_type,source_class,waterpoint_type,waterpoint_type_group,status_group,dataset
0,69572,6000.0,2011-03-14,Roman,1390,Roman,34.938093,-9.856322,none,0,...,good,enough,enough,spring,spring,groundwater,communal standpipe,communal standpipe,functional,train
1,8776,0.0,2013-03-06,Grumeti,1399,GRUMETI,34.698766,-2.147466,Zahanati,0,...,good,insufficient,insufficient,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional,train
2,34310,25.0,2013-02-25,Lottery Club,686,World vision,37.460664,-3.821329,Kwa Mahundi,0,...,good,enough,enough,dam,dam,surface,communal standpipe multiple,communal standpipe,functional,train
3,67743,0.0,2013-01-28,Unicef,263,UNICEF,38.486161,-11.155298,Zahanati Ya Nanyumbu,0,...,good,dry,dry,machine dbh,borehole,groundwater,communal standpipe multiple,communal standpipe,non functional,train
4,19728,0.0,2011-07-13,Action In A,0,Artisan,31.130847,-1.825359,Shuleni,0,...,good,seasonal,seasonal,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional,train


This initial check confirms that the merge worked correctly and gives an overview of the data structure before preprocessing.

### Exploration

Initial review of data types, redundant columns, and category distributions.

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74250 entries, 0 to 74249
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     74250 non-null  int64  
 1   amount_tsh             74250 non-null  float64
 2   date_recorded          74250 non-null  object 
 3   funder                 69743 non-null  object 
 4   gps_height             74250 non-null  int64  
 5   installer              69718 non-null  object 
 6   longitude              74250 non-null  float64
 7   latitude               74250 non-null  float64
 8   wpt_name               74248 non-null  object 
 9   num_private            74250 non-null  int64  
 10  basin                  74250 non-null  object 
 11  subvillage             73780 non-null  object 
 12  region                 74250 non-null  object 
 13  region_code            74250 non-null  int64  
 14  district_code          74250 non-null  int64  
 15  lg

Some variables appear to contain the same information. They will be removed in the cleaning step.

In [33]:
# Apparently duplicated variables
redundant_columns = [
    'scheme_management', 'scheme_name',
    'management', 'management_group',
    'extraction_type', 'extraction_type_group', 'extraction_type_class',
    'water_quality', 'quality_group',
    'quantity', 'quantity_group',
    'source', 'source_type', 'source_class',
    'payment', 'payment_type',
    'waterpoint_type', 'waterpoint_type_group'
]

df[redundant_columns].nunique()

# Inspect the categories
for var in redundant_columns:
    print(f"\nVariable: {var}")
    print(df[df['dataset'] == 'train'][var].value_counts(dropna=False))


Variable: scheme_management
scheme_management
VWC                 36793
WUG                  5206
NaN                  3878
Water authority      3153
WUA                  2883
Water Board          2748
Parastatal           1680
Private operator     1063
Company              1061
Other                 766
SWC                    97
Trust                  72
Name: count, dtype: int64

Variable: scheme_name
scheme_name
NaN                     28810
K                         682
Borehole                  546
Chalinze wate             405
M                         400
                        ...  
Mradi wa maji Vijini        1
Villagers                   1
Magundi water supply        1
Saadani Chumv               1
Mtawanya                    1
Name: count, Length: 2696, dtype: int64

Variable: management
management
vwc                 40507
wug                  6515
water board          2933
wua                  2535
private operator     1971
parastatal           1768
water authority      

Several variables represent the same information at different levels of aggregation, so the redundant ones are removed to reduce dimensionality and avoid multicollinearity.

The following variables are kept because they offer the best balance between detail and generalization:

`extraction_type_class`, `management_group`, `payment_type`, `quality_group`, `quantity`, `scheme_management`, `source_type`, `waterpoint_type_group`.

### Cleaning

Redundant or uninformative columns are removed and missing values are handled.

In [34]:
# Drop redundant and unnecessary columns
columns_drop = [
    'scheme_name', 'management', 'extraction_type', 'extraction_type_group',
    'water_quality', 'quantity_group', 'source', 'source_class',
    'payment', 'waterpoint_type',
    'num_private', 'wpt_name', 'recorded_by', 'subvillage',
]
df.drop(columns=columns_drop, inplace=True, errors='ignore')

In [35]:
# Explore categories and values of the remaining columns
for i in df.columns:
    print(f"\nVariable: {i}")
    print(df[i].value_counts())


Variable: id
id
69572    1
51488    1
40697    1
40221    1
68749    1
        ..
66431    1
70261    1
26783    1
46768    1
68707    1
Name: count, Length: 74250, dtype: int64

Variable: amount_tsh
amount_tsh
0.0         52049
500.0        3874
50.0         3103
1000.0       1858
20.0         1812
            ...  
250000.0        1
350000.0        1
5400.0          1
53.0            1
0.5             1
Name: count, Length: 102, dtype: int64

Variable: date_recorded
date_recorded
2011-03-17    695
2011-03-15    694
2013-02-03    684
2011-03-16    650
2011-03-14    649
             ... 
2011-09-01      1
2011-09-28      1
2011-09-16      1
2011-09-13      1
2004-01-09      1
Name: count, Length: 369, dtype: int64

Variable: funder
funder
Government Of Tanzania    11299
Danida                     3907
Hesawa                     2782
Rwssp                      1703
World Bank                 1701
                          ...  
Sido                          1
Mosqure                   

- **id**: unique record identifier only; adds no predictive information.
- **amount_tsh**: highly skewed with many zeros; reflects a fee but has little useful variability.
- **date_recorded**: time variable that can be turned into features (year, month); not used raw.
- **funder**: high cardinality, many unique values.
- **gps_height**: geographically useful, but with many zeros (possible missing values).
- **installer**: high cardinality and noisy categories.
- **longitude / latitude**: key geographic variables; kept.
- **basin / region / region_code / district_code / lga / ward**: hierarchical geographic variables, some redundant with each other.
- **population**: many zeros and extreme values.
- **public_meeting / permit**: useful boolean variables; kept.
- **construction_year**: many zeros (implicit missing values); transformed into age.
- **extraction_type_class**: summarized, useful version of the extraction type.
- **management_group**: grouped, more stable version of management.
- **payment_type**: clean, grouped version of payment.
- **quality_group**: water quality summary; reduces noise.
- **quantity**: important variable on water availability.
- **source_type**: appropriate intermediate level for the water source.
- **waterpoint_type_group**: useful grouping of the waterpoint type.
- **status_group**: target variable.
- **dataset**: train/test indicator, used only to control the split.

In [36]:
# Total NaN per variable
df.isnull().sum()

id                           0
amount_tsh                   0
date_recorded                0
funder                    4507
gps_height                   0
installer                 4532
longitude                    0
latitude                     0
basin                        0
region                       0
region_code                  0
district_code                0
lga                          0
ward                         0
population                   0
public_meeting            4155
scheme_management         4847
permit                    3793
construction_year            0
extraction_type_class        0
management_group             0
payment_type                 0
quality_group                0
quantity                     0
source_type                  0
waterpoint_type_group        0
status_group             14850
dataset                      0
dtype: int64

The variables `scheme_management`, `installer` and `funder` contain missing values, which are imputed with a new category called `'unknown'`, since missing information in these categorical variables can be informative in itself.

For `public_meeting` and `permit`, missing values are imputed with the mode.

The target `status_group` keeps its missing values in the test set, since that is precisely the variable to predict.

In [37]:
# Handle NaNs
df['scheme_management'].fillna('unknown', inplace=True)
df['installer'].fillna('unknown', inplace=True)
df['funder'].fillna('unknown', inplace=True)
df['public_meeting'].fillna(True, inplace=True)
df['permit'].fillna(True, inplace=True)
# df.isnull().sum()  # Post-imputation check

Missing values have been imputed.

### Feature Engineering

New variables are created from the existing information to improve the model's predictive power. The process has three blocks: temporal features, geospatial features, and binary features/transformations.

#### 1 Temporal Features

In [38]:
# Convert date_recorded to datetime and extract components
df['date_recorded'] = pd.to_datetime(df['date_recorded'])
df['year_recorded']  = df['date_recorded'].dt.year
df['month_recorded'] = df['date_recorded'].dt.month

# Compute PUMP AGE
# Construction year 0 = unknown, treated as NaN
df['construction_year_clean'] = df['construction_year'].replace(0, np.nan)
df['pump_age'] = df['year_recorded'] - df['construction_year_clean']
# Older pumps are more likely to fail

# Bin the age
df['pump_age_bin'] = pd.cut(
    df['pump_age'],
    bins=[-1, 5, 10, 20, 30, 100],
    labels=['0-5', '6-10', '11-20', '21-30', '30+']
)

print('Pump age distribution:')
print(df[df['dataset'] == 'train']['pump_age_bin'].value_counts())

# Drop date_recorded: no longer needed after extracting features
df.drop(columns=['date_recorded'], inplace=True, errors='ignore')

Pump age distribution:
pump_age_bin
0-5      11630
11-20     9456
30+       6336
21-30     5633
6-10      5627
Name: count, dtype: int64


The year and month of recording were extracted from `date_recorded` to capture age as a feature. Pump age (`pump_age`) is computed by subtracting the construction year from the recording year, treating `0` values in `construction_year` as unknown (NaN). Older pumps are expected to be more likely to fail.

Age is also grouped into ranges (`pump_age_bin`) to capture the relationship by age bracket.

#### 2 Geospatial Features

In [39]:
# Approximate center of Tanzania (Dodoma)
TANZANIA_CENTER_LAT = -6.1722
TANZANIA_CENTER_LON = 35.7395

# Euclidean distance to the country's center
df['dist_to_center'] = np.sqrt(
    (df['latitude']  - TANZANIA_CENTER_LAT)**2 +
    (df['longitude'] - TANZANIA_CENTER_LON)**2
)

# lat × lon interaction as a composite feature
df['lat_lon_product'] = df['latitude'] * df['longitude']

# Regional pump density (number of pumps in the same region)
region_density = df.groupby('region')['id'].transform('count')
df['region_pump_density'] = region_density

# Squared altitude (captures non-linear relationship with altitude)
df['gps_height_sq'] = df['gps_height'] ** 2

New variables are created to improve predictive power by capturing geographic and non-linear patterns:

`dist_to_center`: distance from each point to the approximate center of Tanzania (Dodoma), capturing global spatial dependence.      
`lat_lon_product`: interaction between latitude and longitude to capture non-linear spatial relationships.      
`region_pump_density`: pump density per region, reflecting demand pressure and existing infrastructure.    
`gps_height_sq`: quadratic transformation of altitude to model non-linear elevation effects.       

Another important change to improve the model.

#### 3 Binary Features and Transformations

In [40]:
# Binary indicators for the presence of relevant information
df['amount_tsh_bin']        = (df['amount_tsh'] > 0).astype(int)
df['has_construction_year'] = (df['construction_year'] > 0).astype(int)
df['has_population']        = (df['population'] > 0).astype(int)

# Log transform of amount_tsh to reduce the effect of outliers
df['amount_tsh_log'] = np.log1p(df['amount_tsh'])

Feature Engineering (binary and transformations)

New variables are created to improve data quality and reduce scale and skewness issues:

`amount_tsh_bin`: indicates whether there is a water fee (> 0). Captures the difference between paid and free wells.       
`has_construction_year`: indicates whether the construction year is valid (> 0), since 0 means missing.       
`has_population`: indicates whether a population value is recorded (> 0), helping to separate true missing data.       

`amount_tsh_log`: log transformation of the water fee to reduce the impact of outliers and extreme skewness.        

These variables help the model handle implicit missing values, skewness, and outliers.

### Target Encoding

Target Encoding is applied to high-cardinality categorical variables. To avoid **data leakage**, the train and test sets are split first, and the encoder is fitted on train only.

In [41]:
# Split train and test BEFORE encoding to avoid leakage
df_train = df[df['dataset'] == 'train'].copy()
df_test  = df[df['dataset'] == 'test'].copy()

# Encode the target for use in target encoding
le = LabelEncoder()
df_train['status_group_enc'] = le.fit_transform(df_train['status_group'])

# Candidate variables for Target Encoding (high cardinality)
target_enc_cols = [
    'installer', 'funder', 'ward', 'lga',
    'scheme_management', 'extraction_type_class',
    'payment_type', 'source_type', 'basin', 'region'
]

# Target Encoder with smoothing to avoid overfitting on rare categories
te = TargetEncoder(cols=target_enc_cols, smoothing=10)

# Fit on train
df_train[target_enc_cols] = te.fit_transform(
    df_train[target_enc_cols],
    df_train['status_group_enc']
)

# Apply to test (transform only, no refit)
df_test[target_enc_cols] = te.transform(df_test[target_enc_cols])

print('Target Encoding applied to:', target_enc_cols)

Target Encoding applied to: ['installer', 'funder', 'ward', 'lga', 'scheme_management', 'extraction_type_class', 'payment_type', 'source_type', 'basin', 'region']


**Target Encoding** converts high-cardinality categorical variables into numeric values based on the mean of the target.

Process:

- The data is split into **train and test** to avoid data leakage.
- The target `status_group` is encoded as numeric values.
- High-cardinality categorical variables are selected.
- For each category, the target mean is computed on the training set.
- Each category is replaced by that mean.
- On test, only the transformation learned on train is applied.

**Smoothing** is used to avoid overfitting on rare categories.

Result: categorical variables become informative numeric variables for the model.

### Prepare X_train, y_train, X_test

The model input matrices are prepared: One-Hot Encoding for low-cardinality variables, column alignment between train and test, and definition of the final features.

In [42]:
# Variables to exclude from the model
drop_cols = ['dataset', 'status_group', 'status_group_enc', 'id',
             'construction_year',   # Replaced by pump_age
             'pump_age_bin']        # Numeric pump_age is used instead

# Convert booleans to int
for col in df_train.select_dtypes(include='bool').columns:
    df_train[col] = df_train[col].astype(int)
    df_test[col]  = df_test[col].astype(int)

# One-Hot Encoding for low-cardinality variables
low_card_cols = ['quality_group', 'quantity', 'waterpoint_type_group', 'management_group']
df_train = pd.get_dummies(df_train, columns=[c for c in low_card_cols if c in df_train.columns])
df_test  = pd.get_dummies(df_test,  columns=[c for c in low_card_cols if c in df_test.columns])

# Align train/test columns (in case get_dummies creates different columns)
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

# Define X and y
feature_cols = [c for c in df_train.columns if c not in drop_cols]
# Drop columns that are still object type
feature_cols = [c for c in feature_cols if df_train[c].dtype != 'object']

X            = df_train[feature_cols].fillna(-999)
y            = le.fit_transform(df_train['status_group'])
X_test_final = df_test[feature_cols].fillna(-999)

print(f'Shape X: {X.shape} | Shape X_test: {X_test_final.shape}')
print('Classes:', le.classes_)

Shape X: (59400, 53) | Shape X_test: (14850, 53)
Classes: ['functional' 'functional needs repair' 'non functional']


Final preprocessing before training:

- Unused or redundant columns are dropped (`id`, original variables already transformed, and the raw target).
- Boolean variables are converted to numeric (0/1).
- **One-Hot Encoding** is applied to low-cardinality categorical variables to avoid imposing an order.
- Train and test columns are aligned to ensure consistency.
- The following are defined:
  - **X**: predictors
  - **y**: encoded target
  - **X_test_final**: test set ready for prediction
- Missing values are filled with -999 so tree-based models can use them.

The result is a fully numeric, clean dataset ready for training.

### LightGBM with Optuna

LightGBM hyperparameters are optimized with Bayesian search using Optuna (50 trials).

In [43]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def objective_lgbm(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1000),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    model = lgb.LGBMClassifier(**params, random_state=42)
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

# 50 trials — takes ~10-15 min
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'\nBest LGBM accuracy: {study_lgbm.best_value:.4f}')
print(f'Best parameters: {study_lgbm.best_params}')

  0%|          | 0/50 [00:00<?, ?it/s]


Best LGBM accuracy: 0.8189
Best parameters: {'n_estimators': 920, 'max_depth': 8, 'learning_rate': 0.04141240158548312, 'num_leaves': 80, 'min_child_samples': 26, 'subsample': 0.8403516273965683, 'colsample_bytree': 0.6456485963913453, 'reg_alpha': 0.027479202747828498, 'reg_lambda': 2.0273328683251975e-07}


A multiclass classification model is trained with **LightGBM**, with hyperparameters tuned by **Optuna**:

- The dataset is split into training and validation (80/20), stratified to preserve the class distribution.
- An objective function is defined that:
  - Trains a LightGBM model.
  - Evaluates performance with **accuracy** on validation.
- Optuna explores different hyperparameter combinations (depth, learning rate, regularization, etc.).
- Overfitting is controlled with:
  - early stopping
  - feature and row subsampling
- 50 trials are run to find the best configuration.

Result:
- An optimized model with ~0.82 validation accuracy.
- The best hyperparameters are selected for the final model.

In [44]:
# Train the best LGBM model on ALL training data
best_lgbm = lgb.LGBMClassifier(**study_lgbm.best_params, random_state=42, verbosity=-1)
best_lgbm.fit(X, y)

# Cross-validation to estimate the real score
cv_lgbm = cross_val_score(best_lgbm, X, y, cv=5, scoring='accuracy')
print(f'CV LGBM: {cv_lgbm.mean():.4f} ± {cv_lgbm.std():.4f}')

CV LGBM: 0.8186 ± 0.0022


The final model is trained with the best hyperparameters found:

- **LightGBM** is retrained on the full training set.
- Performance is evaluated with **5-fold cross-validation** for a more robust estimate.
- Accuracy is the main metric.

Result:
- Mean CV accuracy: **0.817**
- Low standard deviation (~0.002), indicating a stable model.

This step confirms that the model generalizes consistently before generating the final predictions.

In [45]:
# Evaluate LGBM on validation
y_val_pred_lgbm = best_lgbm.predict(X_val)
print('LGBM')
print(classification_report(y_val, y_val_pred_lgbm, target_names=le.classes_))

LGBM
                         precision    recall  f1-score   support

             functional       0.88      0.96      0.92      6452
functional needs repair       0.86      0.59      0.70       863
         non functional       0.94      0.87      0.90      4565

               accuracy                           0.90     11880
              macro avg       0.89      0.81      0.84     11880
           weighted avg       0.90      0.90      0.90     11880



The **LightGBM** model reaches an **accuracy of 0.89** on the validation set, reflecting good overall performance. It performs strongly on the *functional* class, with high detection ability, and on *non functional*, with good precision and recall. Its main weakness is the *functional needs repair* class, where recall is 0.55, showing difficulty in identifying this intermediate category. Overall, the model works well but needs improvement on the minority class.

#### LGBM Predictions

In [46]:
# Map indices to labels
label_mapping = {0: 'functional', 1: 'functional needs repair', 2: 'non functional'}

predictions_lgbm = best_lgbm.predict(X_test_final)

submission_lgbm = pd.DataFrame({
    'id': df_test['id'],
    'status_group': [label_mapping[p] for p in predictions_lgbm]
})
print(submission_lgbm.shape)

# Save the file
submission_lgbm.to_csv('submission_lgbm.csv', index=False, encoding='utf-8')
print('\nFile saved: submission_lgbm.csv')
print(submission_lgbm['status_group'].value_counts())

(14850, 2)

File saved: submission_lgbm.csv
status_group
functional                 8955
non functional             5315
functional needs repair     580
Name: count, dtype: int64


The final predictions are generated with the trained model:

- The **LightGBM** model is applied to the test set (`X_test_final`).
- The model's numeric output is mapped back to the original labels.
- A DataFrame is built with:
  - `id`: record identifier
  - `status_group`: final predicted waterpoint condition

Result:
- A predictions file ready for submission or evaluation.
- The output shape is checked to confirm consistency with the test set.

### CatBoost with Optuna

A CatBoost model is optimized and combined with LGBM by averaging probabilities (ensemble).

In [47]:
def objective_catboost(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'loss_function': 'MultiClass',
        'eval_metric': 'Accuracy',
        'verbose': False,
        'random_seed': 42
    }
    
    model = CatBoostClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
    
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

# Run the optimization
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_catboost, n_trials=30, show_progress_bar=True)

print(f'\nBest CatBoost accuracy: {study_cat.best_value:.4f}')

  0%|          | 0/30 [00:00<?, ?it/s]


Best CatBoost accuracy: 0.8170


A **CatBoost** model is trained with hyperparameters tuned by **Optuna**:

- An objective function trains the model and evaluates it on validation.
- Key hyperparameters are tuned, such as depth, learning rate, regularization, and sampling.
- **Early stopping** is used to avoid overfitting.
- **Accuracy** is maximized.

Result:
- 30 optimization trials are run.
- The best validation accuracy is approximately **0.815**.

This model is used as an alternative to LightGBM for performance comparison.

In [54]:
# Map indices to labels
label_mapping = {0: 'functional', 1: 'functional needs repair', 2: 'non functional'}

# CatBoost predictions
predictions_cat = best_cat.predict(X_test_final).ravel().astype(int)

# Submission DataFrame
submission_cat = pd.DataFrame({
    'id': df_test['id'],
    'status_group': [label_mapping[p] for p in predictions_cat]
})

print(submission_cat.shape)

# Save the file
submission_cat.to_csv('submission_cat.csv', index=False, encoding='utf-8')

print('\nFile saved: submission_cat.csv')
print(submission_cat['status_group'].value_counts())

(14850, 2)

File saved: submission_cat.csv
status_group
functional                 8960
non functional             5329
functional needs repair     561
Name: count, dtype: int64


The final **CatBoost** model is trained with the best hyperparameters from Optuna on the full training set:

- The model is retrained on the **full X and y** to maximize learning.
- Predictions are generated on the test set.
- Numeric predictions are mapped back to the original labels.
- The submission file is built with:
  - `id`: record identifier
  - `status_group`: final prediction

Finally, the file is exported as CSV, ready for evaluation in the competition.

In [53]:
# Evaluate CatBoost on validation only
y_val_pred_cat = best_cat.predict(X_val)

print('CatBoost')
print(classification_report(y_val, y_val_pred_cat, target_names=le.classes_))

CatBoost
                         precision    recall  f1-score   support

             functional       0.87      0.95      0.91      6452
functional needs repair       0.84      0.54      0.66       863
         non functional       0.92      0.86      0.89      4565

               accuracy                           0.89     11880
              macro avg       0.88      0.78      0.82     11880
           weighted avg       0.89      0.89      0.88     11880



The **CatBoost** model reaches an **accuracy of 0.88** on the validation set, showing good overall performance, slightly below LightGBM. It performs strongly on the *functional* class, with high detection ability, and keeps good precision and recall on *non functional*. Its main weakness is again the *functional needs repair* class, with a recall of 0.51, showing difficulty in identifying this intermediate category. Overall, it is a competitive model, but less sensitive to the minority class.

#### Model Comparison (LightGBM vs CatBoost)

Both models perform very similarly, with **LightGBM slightly ahead (0.89 accuracy)** of **CatBoost (0.88 accuracy)**. On the *functional* class, performance is practically identical, with a recall of 0.95 in both cases, reflecting excellent detection of functional wells. On *non functional*, LightGBM has a slight edge in precision and overall balance. The biggest difference is on *functional needs repair*, where both models struggle, but LightGBM achieves slightly higher recall (0.55 vs 0.51), making it more sensitive to this minority class. Overall, **LightGBM is slightly better**, although both models are highly competitive.